# Exploration 03 — standardization, target transform and dimension (X2: Track A and Track B)

> **Exploratory** (CLAUDE.md, "Exploration"). **Inputs:** the headline-legal molecule-level feature sets from
> notebook 02 (Z and R only). **Data:** cross-validation inside the training sets S_(s,N) for Track B, and the
> development set (`dev`) for Track A. No test set is read.

**Part 3 of the classical analysis.** Input: `data/processed/explore_features.parquet` and the feature catalog
(notebook 02). Output: `results/explore03_preprocessing.json`, read by notebooks 04 and 05: the feature scaling and
target transform for each track, and the reduction method and number of inputs k for the quantum-comparable models.

X1 chose Yeo-Johnson scaling and a √|μ| target at N ≤ 1000 on its old features. X2 re-tests both, because the
feature sets changed (up to 188 features) and Track A trains on up to 99k molecules, where a choice that helped a
small model may not help a large one. The latent-charge model standardizes its own per-atom inputs and predicts
|μ| directly, so these choices concern the molecule-level (tabular) models.

**Choice rules, fixed before any number below was computed:**
- **Track B scaling and target:** the candidate with the lowest mean 5-fold CV MAE over every paired (model,
  feature set, seed, N ≤ 1000) cell; all candidates share the cells and folds (the X1 rule).
- **Track A scaling and target:** the candidate with the lowest mean development-set MAE over the Track A cells
  (models fit on S_(0,N) for large N, tuned inside it).
- **Track B reduction:** for each method (PCA, partial least squares, top-k by mutual information), the mean CV MAE
  of RBF kernel ridge over seeds at each k, at N = 1000. The method with the lowest minimum wins; k is the smallest
  value whose mean MAE is within one standard error (over seeds) of that method's minimum.

Runtime: about 45 minutes. Every cell is safe to re-run.

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline

from qm9dipole import REPO_ROOT
from qm9dipole.analysis import summarize_curves
from qm9dipole.evaluate import dev_curve, learning_curve
from qm9dipole.explore import PREPROCESSING_PATH, feature_sets, load_catalog, load_config, load_features
from qm9dipole.models import classical
from qm9dipole.models.fitters import TabularFitter, XGBFitter
from qm9dipole.plots import INK_SECONDARY, MODEL_COLORS, MODEL_LABELS, MUTED, SERIES, SURFACE, use_style
from qm9dipole.preprocess import scaling_steps
from qm9dipole.provenance import config_hash, git_hash, save_result
from qm9dipole.splits import load_splits

use_style()
FIG_DIR = REPO_ROOT / "figures"
CFG = load_config()
features = load_features()
SETS = feature_sets(load_catalog())
splits = load_splits()
y = features["mu"]
pool = features.loc[splits.pool]
TRACK_B = {s: {n: by_n[n] for n in splits.track_b_sizes} for s, by_n in splits.train.items()}
COMPARE = ("structure+", "all_legal")
DEV = {"dev": splits.dev}
print(f"{len(features):,} development-universe molecules; training pool {len(pool):,}; Track B sizes {splits.track_b_sizes}")
print("feature sets:", ", ".join(f"{s} ({len(SETS[s])})" for s in ("composition", "cm", "structure", "structure+", "all_legal")))

## 1. Why scaling matters here

Ridge's penalty, the RBF kernel's distances, PCA, partial least squares and the quantum feature map's rotation
angles all treat a unit step in every feature as equally important. Raw features differ in scale by orders of
magnitude, so without scaling the largest-variance features would dominate all of them. Trees (XGBoost) split on
one feature at a time and are unaffected by monotone rescaling.

In [ ]:
spread = pool[SETS["all_legal"]].std()
print(f"standard deviation across the {len(spread)} all_legal features: {spread.min():.3g} to {spread.max():.3g} "
      f"(ratio {spread.max() / spread[spread > 0].min():,.0f})")
display(spread.sort_values(ascending=False).iloc[[0, 1, 2, -3, -2, -1]].rename("std").to_frame()
        .style.format("{:.3g}").set_caption("largest and smallest spreads"))

## 2. What each scaling does to the shape

Five candidates (`src/qm9dipole/preprocess.py`): z-score (**standard**), median/IQR (**robust**), log of the skewed
columns then z-score (**log_standard**), a fitted power transform (**yeo_johnson**), and a rank map to a normal
(**quantile_normal**). Fit on the pool for description only (models refit inside every fold). The share of values
beyond ±3 matters for the quantum feature map: angles wrap every 2π, so far-out values alias onto ordinary ones.

In [ ]:
CANDIDATES = CFG["standardization"]["candidates"]
SIGMA = CFG["quantum"]["angle_sigma"]
legal_X = pool[SETS["all_legal"]].to_numpy()
shape_rows, scaled_pool = [], {}
for sc in CANDIDATES:
    Z = Pipeline(scaling_steps(sc, CFG["standardization"]["skew_threshold"])).fit_transform(legal_X)
    scaled_pool[sc] = Z
    sk = pd.DataFrame(Z).skew().abs()
    shape_rows.append({"scaling": sc, "median |skew|": sk.median(), "max |skew|": sk.max(),
                       "max |value|": np.abs(Z).max(), f"share beyond ±{SIGMA}": (np.abs(Z) > SIGMA).mean()})
shape = pd.DataFrame(shape_rows).set_index("scaling")
display(shape.style.format({"median |skew|": "{:.2f}", "max |skew|": "{:.1f}", "max |value|": "{:.1f}",
                            f"share beyond ±{SIGMA}": "{:.2%}"}))

show = ["inertia_moment_3", "qeq_dipole_D", "bond_dipole_norm", "n_N"]
idx = [SETS["all_legal"].index(c) for c in show]
fig, axes = plt.subplots(len(show), len(CANDIDATES), figsize=(15, 8.4))
for i, (col, j) in enumerate(zip(show, idx)):
    for k, sc in enumerate(CANDIDATES):
        ax = axes[i, k]
        ax.hist(scaled_pool[sc][:, j], bins=50, color=SERIES[0], linewidth=0)
        ax.tick_params(labelsize=7)
        ax.grid(axis="x", visible=False)
        if i == 0:
            ax.set_title(sc, fontsize=10)
        if k == 0:
            ax.set_ylabel(col, fontsize=9)
fig.suptitle("Four legal features under each scaling (rows: features; columns: scalings)", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "explore03_scaling_shapes.png")
plt.show()
del scaled_pool

## 3. Choose the scaling

**Track B** (N ≤ 1000): ridge and RBF kernel ridge on structure+ and all_legal, 5-fold CV inside every S_(s,N), the
same folds for every candidate. **Track A** (large N): ridge fit on S_(0,10000) and on the full pool, and RBF kernel
ridge on S_(0,3000), each tuned on an inner validation split of its training set, scored on the development set.
Targets are standardized here; the target transform is chosen next.

In [ ]:
start = time.perf_counter()
frames = {name: features[SETS[name]] for name in COMPARE}
runs = []
for sc in CANDIDATES:
    def make_specs(seed, fname, sc=sc):
        return {m: classical.build(m, seed, scaling=sc, n_features=len(SETS[fname])) for m in ("ridge", "rbf_krr")}
    res, _ = learning_curve(make_specs, frames, y, TRACK_B, progress=False)
    runs.append(res.assign(scaling=sc))
    print(f"Track B {sc:16s} done ({(time.perf_counter() - start) / 60:.1f} min)", flush=True)
scaling_b = pd.concat(runs, ignore_index=True)

FULL = len(splits.pool)
A_CELLS = {("ridge", 10000), ("ridge", FULL), ("rbf_krr", 3000)}
runs = []
for sc in CANDIDATES:
    def make_fitters(seed, n, sc=sc):
        out = {}
        for fname in COMPARE:
            for m in ("ridge", "rbf_krr"):
                if (m, n) in A_CELLS:
                    est, grid = classical.build(m, seed, scaling=sc, n_features=len(SETS[fname]))
                    out[f"{m}|{fname}"] = TabularFitter(features[SETS[fname]], est, grid, seed)
        return out
    res, _ = dev_curve(make_fitters, {0: {n: splits.train[0][n] for n in (3000, 10000, FULL)}}, DEV, y, progress=False)
    runs.append(res.assign(scaling=sc))
    print(f"Track A {sc:16s} done ({(time.perf_counter() - start) / 60:.1f} min)", flush=True)
scaling_a = pd.concat(runs, ignore_index=True)
scaling_a[["model", "features"]] = scaling_a["model"].str.split("|", expand=True)

In [ ]:
def choose(results, by, value="mae_D"):
    paired = results.pivot_table(index=[c for c in ("seed", "n_train", "model", "features") if c in results], columns=by, values=value)
    mean = paired.mean().loc[CANDIDATES] if by == "scaling" else paired.mean()
    return mean, paired

mean_b, paired_b = choose(scaling_b, "scaling")
mean_a, paired_a = choose(scaling_a, "scaling")
best_scaling = {"track_b": mean_b.idxmin(), "track_a": mean_a.idxmin()}
table = pd.DataFrame({"Track B: mean CV MAE (D)": mean_b, "Track B: cells won": paired_b.eq(paired_b.min(axis=1), axis=0).mean(),
                      "Track A: mean dev MAE (D)": mean_a, "Track A: cells won": paired_a.eq(paired_a.min(axis=1), axis=0).mean()})
display(table.style.format({c: "{:.4f}" if "MAE" in c else "{:.0%}" for c in table.columns})
        .set_caption(f"{len(paired_b)} Track B cells, {len(paired_a)} Track A cells"))
display(scaling_b.pivot_table(index=["n_train", "model"], columns="scaling", values="mae_D")[CANDIDATES]
        .style.format("{:.3f}").highlight_min(axis=1, props="font-weight: bold").set_caption("Track B: mean CV MAE (D) by N and model"))
display(scaling_a.pivot_table(index=["model", "n_train", "features"], columns="scaling", values="mae_D")[CANDIDATES]
        .style.format("{:.3f}").highlight_min(axis=1, props="font-weight: bold").set_caption("Track A: development-set MAE (D)"))
print("chosen scaling:", best_scaling)

## 4. Choose the target transform

A model fit on a transformed target is inverted back to debye before scoring, so only accuracy in debye counts.
Every candidate also standardizes, so kernel ridge's missing intercept never biases it toward 0 D. Track B: ridge,
RBF kernel ridge and XGBoost (X1's grid) by 5-fold CV; Track A: ridge on S_(0,10000) and the full pool, kernel ridge
on S_(0,3000) and XGBoost (randomized search, 8 draws) on S_(0,10000), scored on the development set. Scaling as
chosen above, per track.

In [ ]:
TARGETS = CFG["standardization"]["target_candidates"]
start = time.perf_counter()
runs = []
for tg in TARGETS:
    def make_specs(seed, fname, tg=tg):
        return {m: classical.build(m, seed, scaling=best_scaling["track_b"], target=tg, n_features=len(SETS[fname]))
                for m in ("ridge", "rbf_krr", "xgb")}
    res, _ = learning_curve(make_specs, frames, y, TRACK_B, progress=False)
    runs.append(res.assign(target=tg))
    print(f"Track B {tg:9s} done ({(time.perf_counter() - start) / 60:.1f} min)", flush=True)
target_b = pd.concat(runs, ignore_index=True)

A_CELLS_T = {("ridge", 10000), ("ridge", FULL), ("rbf_krr", 3000), ("xgb", 10000)}
runs = []
for tg in TARGETS:
    def make_fitters(seed, n, tg=tg):
        out = {}
        for fname in COMPARE:
            for m in ("ridge", "rbf_krr", "xgb"):
                if (m, n) not in A_CELLS_T:
                    continue
                if m == "xgb":
                    out[f"{m}|{fname}"] = XGBFitter(features[SETS[fname]], seed, n_iter=8, target=tg)
                else:
                    est, grid = classical.build(m, seed, scaling=best_scaling["track_a"], target=tg, n_features=len(SETS[fname]))
                    out[f"{m}|{fname}"] = TabularFitter(features[SETS[fname]], est, grid, seed)
        return out
    res, _ = dev_curve(make_fitters, {0: {n: splits.train[0][n] for n in (3000, 10000, FULL)}}, DEV, y, progress=False)
    runs.append(res.assign(target=tg))
    print(f"Track A {tg:9s} done ({(time.perf_counter() - start) / 60:.1f} min)", flush=True)
target_a = pd.concat(runs, ignore_index=True)
target_a[["model", "features"]] = target_a["model"].str.split("|", expand=True)

tb = target_b.groupby("target")[["mae_D", "rmse_D"]].mean().loc[TARGETS]
ta = target_a.groupby("target")[["mae_D", "rmse_D"]].mean().loc[TARGETS]
best_target = {"track_b": tb["mae_D"].idxmin(), "track_a": ta["mae_D"].idxmin()}
display(pd.concat({"Track B (CV)": tb, "Track A (dev)": ta}, axis=1).style.format("{:.4f}")
        .set_caption("mean MAE and RMSE (D) by target transform"))
display(target_b.pivot_table(index=["n_train", "model"], columns="target", values="mae_D")[TARGETS]
        .style.format("{:.3f}").highlight_min(axis=1, props="font-weight: bold").set_caption("Track B: mean CV MAE (D)"))
display(target_a.pivot_table(index=["model", "n_train", "features"], columns="target", values="mae_D")[TARGETS]
        .style.format("{:.3f}").highlight_min(axis=1, props="font-weight: bold").set_caption("Track A: development-set MAE (D)"))
print("chosen target transform:", best_target)

## 5. Track B: how many inputs, and which?

The quantum model takes one input per qubit, so the matched classical model must too. Three ways to reduce the 188
all_legal features to k inputs, each fit inside the pipeline on every fold and followed by re-standardization (the
same numbers an angle encoding would see):
- **PCA:** the k directions of largest variance (unsupervised). X1 found the components that track |μ| outside the
  first 8;
- **PLS** (partial least squares): the k directions of largest covariance with the target (supervised);
- **top-k by mutual information** with the target (supervised; keeps original, interpretable features).

RBF kernel ridge (the classical twin of a quantum kernel model) by 5-fold CV inside every S_(s,N), N ≤ 1000, for
k from 2 to 32; the uncompressed model is the reference.

In [ ]:
KS = [2, 4, 6, 8, 10, 12, 16, 24, 32]
METHODS = list(classical.REDUCTIONS)
frames_k = {f"{m}/{k}": features[SETS["all_legal"]] for m in METHODS for k in KS}
frames_k["all_legal (no reduction)"] = features[SETS["all_legal"]]

def make_specs(seed, fname):
    if "/" not in fname:
        return {"rbf_krr": classical.build("rbf_krr", seed, scaling=best_scaling["track_b"], target=best_target["track_b"],
                                           n_features=len(SETS["all_legal"]))}
    m, k = fname.split("/")
    return {"rbf_krr": classical.build("rbf_krr", seed, scaling=best_scaling["track_b"], target=best_target["track_b"],
                                       reduction=(m, int(k)))}

start = time.perf_counter()
kcurve, _ = learning_curve(make_specs, frames_k, y, TRACK_B, progress=False)
print(f"{len(kcurve)} tuned CV runs in {(time.perf_counter() - start) / 60:.1f} min")
kcurve["method"] = kcurve["features"].map(lambda f: f.split("/")[0] if "/" in f else "none")
kcurve["k"] = kcurve["features"].map(lambda f: int(f.split("/")[1]) if "/" in f else len(SETS["all_legal"]))
ks = kcurve.groupby(["n_train", "method", "k"])["mae_D"].agg(["mean", "std", "count"]).reset_index()
ks["se"] = ks["std"] / np.sqrt(ks["count"])

N_CHOICE = max(splits.track_b_sizes)
at = ks[(ks["n_train"] == N_CHOICE) & (ks["method"] != "none")]
choice = {}
for m, g in at.groupby("method"):
    best = g.loc[g["mean"].idxmin()]
    k_star = int(g.loc[g["mean"] <= best["mean"] + best["se"], "k"].min())
    choice[m] = {"min_mae": float(best["mean"]), "k_at_min": int(best["k"]), "k_one_se": k_star,
                 "mae_at_k_one_se": float(g.loc[g["k"] == k_star, "mean"].iloc[0])}
best_method = min(choice, key=lambda m: choice[m]["min_mae"])
reduction = {"method": best_method, "k": choice[best_method]["k_one_se"], "chosen_at_n": N_CHOICE, "per_method": choice}
display(pd.DataFrame(choice).T.style.format({"min_mae": "{:.3f}", "mae_at_k_one_se": "{:.3f}"})
        .set_caption(f"N = {N_CHOICE}: per method, the best k and the smallest k within one standard error"))
print(f"chosen reduction: {best_method}, k = {reduction['k']}")

fig, axes = plt.subplots(1, len(splits.track_b_sizes), figsize=(15, 4), sharey=False)
for ax, n in zip(axes, splits.track_b_sizes):
    for color, m in zip(SERIES, METHODS):
        g = ks[(ks["n_train"] == n) & (ks["method"] == m)]
        ax.errorbar(g["k"], g["mean"], yerr=g["std"], color=color, marker="o", capsize=2, label=m)
    ref = ks[(ks["n_train"] == n) & (ks["method"] == "none")]["mean"].iloc[0]
    ax.axhline(ref, color=INK_SECONDARY, linewidth=1, linestyle="--")
    ax.annotate(f"all {len(SETS['all_legal'])} features", (KS[-1], ref), xytext=(-4, 4), textcoords="offset points",
                ha="right", color=INK_SECONDARY, fontsize=8)
    ax.axvline(8, color=MUTED, linewidth=0.8)
    ax.set(xscale="log", xticks=KS, xlabel="inputs k (= qubits for a quantum model)", title=f"N = {n}")
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0f}")
    ax.xaxis.set_minor_locator(plt.NullLocator())
axes[0].set_ylabel("CV MAE (D), RBF kernel ridge, mean ± sd over seeds")
axes[0].legend(title="reduction")
fig.suptitle("The cost of compressing all_legal to k inputs", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "explore03_compression_cost.png")
plt.show()
display(ks[ks["method"] != "none"].pivot_table(index=["method", "k"], columns="n_train", values="mean")
        .style.format("{:.3f}").set_caption("mean CV MAE (D) by method, k and N"))

In [ ]:
# What the chosen reduction keeps, described on S_(0,1000) (top-k features are directly interpretable).
from sklearn.feature_selection import mutual_info_regression

ids = splits.train[0][N_CHOICE]
est, _ = classical.build("ridge", 0, scaling=best_scaling["track_b"], target=best_target["track_b"],
                         reduction=("kbest", reduction["k"]))
est.fit(features.loc[ids, SETS["all_legal"]].to_numpy(), y.loc[ids].to_numpy())
picked = np.array(SETS["all_legal"])[est.regressor_.named_steps["reduce"].get_support()]
print(f"top-{reduction['k']} features by mutual information on S_(0,{N_CHOICE}): {', '.join(picked)}")

## 6. Effective dimension: how many independent directions do the features span?

Four measures on the scaled pool (Track A scaling): the **participation ratio** PR = (Σλ)² / Σλ² of the covariance
eigenvalues (the number of equally strong directions with the same overall spread); **components for 90 / 95 /
99 %** of the variance; and the **condition number** √(λ_max / λ_min).

In [ ]:
from qm9dipole.complexity import components_for_share, condition_number, participation_ratio

dim_rows = []
for name in ("composition", "cm", "structure", "structure+", "all_legal"):
    Z = Pipeline(scaling_steps(best_scaling["track_a"], CFG["standardization"]["skew_threshold"])).fit_transform(pool[SETS[name]])
    lam = np.linalg.eigvalsh(np.cov(Z, rowvar=False))[::-1]
    dim_rows.append({"feature set": name, "features": Z.shape[1], "participation ratio": participation_ratio(lam),
                     "k for 90%": components_for_share(lam, 0.90), "k for 95%": components_for_share(lam, 0.95),
                     "k for 99%": components_for_share(lam, 0.99), "condition number": condition_number(lam)})
dims = pd.DataFrame(dim_rows).set_index("feature set")
display(dims.style.format({"participation ratio": "{:.1f}", "condition number": "{:.3g}"}))

## 7. Save the choices for notebooks 04 and 05

In [ ]:
choice_out = {
    "scaling": best_scaling, "target": best_target, "reduction": reduction,
    "rules": {"scaling_and_target": "lowest mean MAE over the paired cells of each track (CV for Track B, dev set for Track A)",
              "reduction": "at the largest Track B size, the method with the lowest minimum mean CV MAE (RBF kernel ridge); "
                           "k = smallest within one standard error over seeds of that minimum"},
    "scaling_mean_mae": {"track_b": mean_b.round(5).to_dict(), "track_a": mean_a.round(5).to_dict()},
    "target_mean_mae": {"track_b": tb["mae_D"].round(5).to_dict(), "track_a": ta["mae_D"].round(5).to_dict()},
    "kbest_features": picked.tolist(),
    "effective_dimension": dims[["features", "participation ratio", "k for 95%"]].round(2).reset_index().to_dict("records"),
    "config_hash": config_hash(CFG), "git_hash": git_hash(),
}
PREPROCESSING_PATH.write_text(json.dumps(choice_out, indent=1) + "\n")
_ = save_result(pd.concat([scaling_b.assign(track="B"), scaling_a.assign(track="A")], ignore_index=True), "explore03_scaling",
                label="exploratory; legal inputs; Track B CV inside S_(s,N); Track A dev set")
_ = save_result(pd.concat([target_b.assign(track="B"), target_a.assign(track="A")], ignore_index=True), "explore03_target",
                label="exploratory; legal inputs; Track B CV inside S_(s,N); Track A dev set")
_ = save_result(kcurve, "explore03_compression_cost", label="exploratory; legal inputs; CV inside S_(s,N), N <= 1000")
print(f"wrote {PREPROCESSING_PATH.relative_to(REPO_ROOT).as_posix()}: scaling {best_scaling}, target {best_target}, "
      f"reduction {reduction['method']} k = {reduction['k']}")

## Summary

*(Written from this run's outputs; see the cells above.)*